# 04 — Dtype Inference Across Engines

Notebook 01 showed that pandas infers every column as `object` because
Metabase wraps comma-formatted numbers in quoted CSV fields. This notebook
asks: **do other engines do better?**

We test four engines on the same raw CSV:

| Engine | Approach |
|--------|----------|
| **pandas** | `read_csv` — baseline (notebook 01) |
| **PyArrow** | `pyarrow.csv.read_csv` — Arrow-native reader, already a dependency |
| **Polars** | `pl.read_csv` / `pl.scan_csv` — Rust-native, Arrow backend |
| **DuckDB** | `read_csv_auto` — SQL engine with an advanced CSV sniffer |

### Why this matters for a data lake

In a cloud data lake (S3/GCS → Parquet/Delta/Iceberg), the **storage format
carries its own schema** — dtype inference happens once at ingestion, not on
every read. The question becomes: which engine handles the **CSV → lake
boundary** most robustly?

At query time, engines like DuckDB, Spark, or Trino read Parquet metadata
directly — no inference needed. The dtype problem is strictly about the
**ingestion step**.

In [1]:
import pandas as pd
import pyarrow.csv as pa_csv
import polars as pl
import duckdb

from fudo.utils.io import RAW_DIR

source_files = sorted(RAW_DIR.glob("SKALAR - Data Revenue_*.csv"))
assert source_files, "No SKALAR revenue CSV found in data/raw/"
SOURCE_FILE = source_files[-1]
print(f"Source: {SOURCE_FILE.name}  ({SOURCE_FILE.stat().st_size / 1024:.0f} KB)")

Source: SKALAR - Data Revenue_2026-04-16T13_53_52.265503937Z.csv  (2231 KB)


## 1. Pandas — baseline

As documented in notebook 01: every column inferred as `object` because of
quoted comma-formatted values.

In [2]:
df_pandas = pd.read_csv(SOURCE_FILE, low_memory=False)

print("=== pandas inferred dtypes ===")
for col in df_pandas.columns:
    print(f"  {col:25s}  {str(df_pandas[col].dtype):15s}  sample: {df_pandas[col].iloc[0]!r}")

=== pandas inferred dtypes ===
  PAIS                       str              sample: 'Chile'
  a.ACCOUNT_ID               str              sample: '349,486'
  FECHA_PRIMER_PAGO          str              sample: 'April 1, 2026'
  PRODUCTO                   str              sample: 'Saas'
  MES                        str              sample: '202,604'
  MONEDA                     str              sample: 'CLP'
  REV_LOCAL                  str              sample: '10,400'


## 2. PyArrow — Arrow-native CSV reader

PyArrow's CSV reader uses Arrow's type inference. It infers from the first
block of rows (default 1 MB) and applies Arrow's type system directly.

Key difference from pandas: Arrow has `int64`, `float64`, `string`,
`timestamp`, etc. as first-class types — there's no `object` catchall.

In [3]:
table_arrow = pa_csv.read_csv(SOURCE_FILE)

print("=== pyarrow inferred types ===")
for field in table_arrow.schema:
    col = table_arrow.column(field.name)
    print(f"  {field.name:25s}  {str(field.type):15s}  sample: {col[0].as_py()!r}")

=== pyarrow inferred types ===
  PAIS                       string           sample: 'Chile'
  a.ACCOUNT_ID               string           sample: '349,486'
  FECHA_PRIMER_PAGO          string           sample: 'April 1, 2026'
  PRODUCTO                   string           sample: 'Saas'
  MES                        string           sample: '202,604'
  MONEDA                     string           sample: 'CLP'
  REV_LOCAL                  string           sample: '10,400'


## 3. Polars — Rust-native engine

Polars uses a configurable `infer_schema_length` (default: 100 rows).
It scans that many rows, then decides the column type.

We test with the default and then with `infer_schema_length=None` (scan all
rows) to see if it makes a difference.

In [4]:
# Default: infer from 100 rows
df_polars_100 = pl.read_csv(SOURCE_FILE, infer_schema_length=100)

print("=== polars (infer_schema_length=100) ===")
for col_name in df_polars_100.columns:
    dtype = df_polars_100[col_name].dtype
    sample = df_polars_100[col_name][0]
    print(f"  {col_name:25s}  {str(dtype):15s}  sample: {sample!r}")

print()

# Full scan: infer from all rows
df_polars_all = pl.read_csv(SOURCE_FILE, infer_schema_length=None)

print("=== polars (infer_schema_length=None, full scan) ===")
for col_name in df_polars_all.columns:
    dtype = df_polars_all[col_name].dtype
    sample = df_polars_all[col_name][0]
    print(f"  {col_name:25s}  {str(dtype):15s}  sample: {sample!r}")

=== polars (infer_schema_length=100) ===
  PAIS                       String           sample: 'Chile'
  a.ACCOUNT_ID               String           sample: '349,486'
  FECHA_PRIMER_PAGO          String           sample: 'April 1, 2026'
  PRODUCTO                   String           sample: 'Saas'
  MES                        String           sample: '202,604'
  MONEDA                     String           sample: 'CLP'
  REV_LOCAL                  String           sample: '10,400'



=== polars (infer_schema_length=None, full scan) ===
  PAIS                       String           sample: 'Chile'
  a.ACCOUNT_ID               String           sample: '349,486'
  FECHA_PRIMER_PAGO          String           sample: 'April 1, 2026'
  PRODUCTO                   String           sample: 'Saas'
  MES                        String           sample: '202,604'
  MONEDA                     String           sample: 'CLP'
  REV_LOCAL                  String           sample: '10,400'


## 4. DuckDB — SQL engine with advanced CSV sniffer

DuckDB's `read_csv_auto` uses a multi-pass sniffer that detects:
- Delimiters, quoting, escaping
- Date/timestamp formats
- **Decimal separators and thousand separators**

In theory, this is the engine most likely to handle the Metabase comma
problem out of the box. Let's see.

In [5]:
con = duckdb.connect()

# Let DuckDB auto-detect everything
result = con.sql(f"""
    SELECT * FROM read_csv_auto('{SOURCE_FILE}')
    LIMIT 5
""")

print("=== duckdb read_csv_auto inferred types ===")
for col_name, col_type in zip(result.columns, result.types):
    print(f"  {col_name:25s}  {str(col_type):15s}")

print("\nSample rows:")
result.show()

=== duckdb read_csv_auto inferred types ===
  PAIS                       VARCHAR        
  a.ACCOUNT_ID               VARCHAR        
  FECHA_PRIMER_PAGO          VARCHAR        
  PRODUCTO                   VARCHAR        
  MES                        VARCHAR        
  MONEDA                     VARCHAR        
  REV_LOCAL                  VARCHAR        

Sample rows:


┌───────────┬──────────────┬────────────────────┬──────────┬─────────┬─────────┬───────────┐
│   PAIS    │ a.ACCOUNT_ID │ FECHA_PRIMER_PAGO  │ PRODUCTO │   MES   │ MONEDA  │ REV_LOCAL │
│  varchar  │   varchar    │      varchar       │ varchar  │ varchar │ varchar │  varchar  │
├───────────┼──────────────┼────────────────────┼──────────┼─────────┼─────────┼───────────┤
│ Chile     │ 349,486      │ April 1, 2026      │ Saas     │ 202,604 │ CLP     │ 10,400    │
│ Colombia  │ 309,479      │ September 25, 2025 │ Saas     │ 202,601 │ COP     │ 131,860   │
│ Chile     │ 349,075      │ March 30, 2026     │ Payments │ 202,604 │ CLP     │ 29.41     │
│ Argentina │ 324,599      │ December 2, 2025   │ Saas     │ 202,604 │ ARS     │ 62,809.92 │
│ Chile     │ 292,587      │ July 11, 2025      │ Saas     │ 202,601 │ CLP     │ 34,200    │
└───────────┴──────────────┴────────────────────┴──────────┴─────────┴─────────┴───────────┘



In [6]:
# Inspect what DuckDB's sniffer actually detected
sniff = con.sql(f"FROM sniff_csv('{SOURCE_FILE}')")
sniff.show()

┌───────────┬─────────┬─────────┬──────────────────┬─────────┬──────────┬───────────┬────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬────────────┬─────────────────┬───────────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│ Delimiter │  Quote  │ Escape  │ NewLineDelimiter │ Comment │ SkipRows │ HasHeader │                                                                                                                              

## 5. Comparison summary — auto-inference

**Result: every engine falls back to string/VARCHAR for every column.**

The Metabase export creates genuine ambiguity: the CSV uses commas as
the field delimiter *and* commas as thousand separators inside quoted
fields. No engine can safely resolve this without hints because
`"349,486"` is indistinguishable from a two-field quoted string to a
sniffer that doesn't know the domain semantics.

In [7]:
# Collect inferred dtypes from each engine
columns = df_pandas.columns.tolist()

pandas_types = {c: str(df_pandas[c].dtype) for c in columns}
arrow_types = {f.name: str(f.type) for f in table_arrow.schema}
polars_types = {c: str(df_polars_all[c].dtype) for c in df_polars_all.columns}

# DuckDB types from the auto-read
duck_full = con.sql(f"SELECT * FROM read_csv_auto('{SOURCE_FILE}') LIMIT 1")
duck_types = dict(zip(duck_full.columns, [str(t) for t in duck_full.types]))

# Protocol expected types
protocol_expected = {
    "PAIS": "str (extra)",
    "a.ACCOUNT_ID": "str",
    "FECHA_PRIMER_PAGO": "datetime",
    "PRODUCTO": "str",
    "MES": "period/date",
    "MONEDA": "str",
    "REV_LOCAL": "float64",
}

comparison_rows = []
for col in columns:
    comparison_rows.append({
        "column": col,
        "expected": protocol_expected.get(col, "?"),
        "pandas": pandas_types.get(col, "—"),
        "pyarrow": arrow_types.get(col, "—"),
        "polars": polars_types.get(col, "—"),
        "duckdb": duck_types.get(col, "—"),
    })

comparison_df = pd.DataFrame(comparison_rows)
comparison_df.style.map(
    lambda v: "background-color: #d9ead3" if v in ("float64", "double", "Float64", "DOUBLE", "f64")
    else "background-color: #d9ead3" if "int" in str(v).lower() or "bigint" in str(v).lower()
    else "",
    subset=["pandas", "pyarrow", "polars", "duckdb"],
)

,column,expected,pandas,pyarrow,polars,duckdb
0,PAIS,str (extra),str,string,String,VARCHAR
1,a.ACCOUNT_ID,str,str,string,String,VARCHAR
2,FECHA_PRIMER_PAGO,datetime,str,string,String,VARCHAR
3,PRODUCTO,str,str,string,String,VARCHAR
4,MES,period/date,str,string,String,VARCHAR
5,MONEDA,str,str,string,String,VARCHAR
6,REV_LOCAL,float64,str,string,String,VARCHAR


## 5.1 Hint-assisted reads — where engines actually differ

Auto-inference failed universally. But each engine offers different
mechanisms to *tell* it how to parse. This is where the ergonomics and
capabilities diverge — and what matters at the CSV → lake ingestion
boundary.

### pandas — manual post-hoc cleaning (what notebook 01 does)

pandas has no `thousands` parameter that works with quoted fields.
The fix is string manipulation after the fact: `.str.replace(",", "").astype(float)`.

In [8]:
# pandas: the thousands= param exists but only works for unquoted fields.
# With quoted fields, we must clean manually (as notebook 01 does).
df_pd_hint = pd.read_csv(SOURCE_FILE, thousands=",", low_memory=False)

print("=== pandas with thousands=',' ===")
for col in ["a.ACCOUNT_ID", "MES", "REV_LOCAL"]:
    print(f"  {col:25s}  {str(df_pd_hint[col].dtype):15s}  sample: {df_pd_hint[col].iloc[0]!r}")

print("\nVerdict: thousands= DOES strip commas from quoted fields in modern pandas."
      "\nBut it's all-or-nothing — it also strips commas from ACCOUNT_ID and MES,"
      "\nwhich are not truly numeric. No column-level control.")

=== pandas with thousands=',' ===
  a.ACCOUNT_ID               int64            sample: np.int64(349486)
  MES                        int64            sample: np.int64(202604)
  REV_LOCAL                  float64          sample: np.float64(10400.0)

Verdict: thousands= DOES strip commas from quoted fields in modern pandas.
But it's all-or-nothing — it also strips commas from ACCOUNT_ID and MES,
which are not truly numeric. No column-level control.


### Polars — explicit schema override

Polars lets you pass a `schema` dict that overrides inference.
Combined with post-read expressions, it's clean and composable.

In [9]:
# Polars: read as String, then cast with expressions — functional and composable
df_pl_clean = (
    pl.read_csv(SOURCE_FILE, schema_overrides={"REV_LOCAL": pl.String})
    .with_columns(
        # Strip commas, then cast
        pl.col("a.ACCOUNT_ID").str.replace_all(",", "").alias("client_id"),
        pl.col("MES").str.replace_all(",", "").alias("mes_clean"),
        pl.col("REV_LOCAL").str.replace_all(",", "").cast(pl.Float64).alias("collections"),
    )
)

print("=== polars with expression-based cleaning ===")
for col in ["client_id", "mes_clean", "collections"]:
    print(f"  {col:25s}  {str(df_pl_clean[col].dtype):15s}  sample: {df_pl_clean[col][0]!r}")

print("\nVerdict: Polars expressions are composable and column-specific.")
print("The lazy API (scan_csv → with_columns → collect) would do this without")
print("loading the full CSV into memory.")

=== polars with expression-based cleaning ===
  client_id                  String           sample: '349486'
  mes_clean                  String           sample: '202604'
  collections                Float64          sample: 10400.0

Verdict: Polars expressions are composable and column-specific.
The lazy API (scan_csv → with_columns → collect) would do this without
loading the full CSV into memory.


### DuckDB — SQL `REPLACE` + `CAST` in the query

DuckDB can clean and cast inline in SQL. It also supports a
`decimal_separator` parameter in newer versions, but the real power
is doing the transform in SQL at read time.

In [10]:
# DuckDB: clean + cast in SQL at read time
duck_clean = con.sql(f"""
    SELECT
        REPLACE("a.ACCOUNT_ID", ',', '')           AS client_id,
        REPLACE("MES", ',', '')                     AS mes_clean,
        CAST(REPLACE("REV_LOCAL", ',', '') AS DOUBLE) AS collections,
        "PAIS"                                      AS pais,
        "MONEDA"                                    AS currency,
        "PRODUCTO"                                  AS product,
        strptime("FECHA_PRIMER_PAGO", '%B %d, %Y') AS fecha_primer_pago
    FROM read_csv_auto('{SOURCE_FILE}')
""")

print("=== duckdb with SQL-based cleaning ===")
for col_name, col_type in zip(duck_clean.columns, duck_clean.types):
    print(f"  {col_name:25s}  {str(col_type):15s}")

print("\nSample:")
duck_clean.limit(5).show()

print("Verdict: DuckDB handles the full transform in a single SQL statement.")
print("This is exactly what a data lake ingestion job looks like —")
print("SELECT ... FROM read_csv → COPY TO 'lake/path.parquet'")

=== duckdb with SQL-based cleaning ===
  client_id                  VARCHAR        
  mes_clean                  VARCHAR        
  collections                DOUBLE         
  pais                       VARCHAR        
  currency                   VARCHAR        
  product                    VARCHAR        
  fecha_primer_pago          TIMESTAMP      

Sample:
┌───────────┬───────────┬─────────────┬───────────┬──────────┬──────────┬─────────────────────┐
│ client_id │ mes_clean │ collections │   pais    │ currency │ product  │  fecha_primer_pago  │
│  varchar  │  varchar  │   double    │  varchar  │ varchar  │ varchar  │      timestamp      │
├───────────┼───────────┼─────────────┼───────────┼──────────┼──────────┼─────────────────────┤
│ 349486    │ 202604    │     10400.0 │ Chile     │ CLP      │ Saas     │ 2026-04-01 00:00:00 │
│ 309479    │ 202601    │    131860.0 │ Colombia  │ COP      │ Saas     │ 2025-09-25 00:00:00 │
│ 349075    │ 202604    │       29.41 │ Chile     │ CLP      

### PyArrow — `ConvertOptions` with custom column types

PyArrow offers `ConvertOptions` for explicit type mapping, but it
cannot do string transformations (strip commas) at read time. You
need a post-read compute step.

In [11]:
import pyarrow as pa
import pyarrow.compute as pc

# PyArrow: read as string, then compute transforms
table = pa_csv.read_csv(SOURCE_FILE)

# Arrow compute kernels for cleaning
client_id = pc.replace_substring(table.column("a.ACCOUNT_ID"), ",", "")
mes_clean = pc.replace_substring(table.column("MES"), ",", "")
collections = pc.cast(
    pc.replace_substring(table.column("REV_LOCAL"), ",", ""),
    pa.float64()
)

print("=== pyarrow with compute kernels ===")
print(f"  client_id     type={client_id.type}   sample: {client_id[0].as_py()!r}")
print(f"  mes_clean     type={mes_clean.type}   sample: {mes_clean[0].as_py()!r}")
print(f"  collections   type={collections.type}  sample: {collections[0].as_py()!r}")

print("\nVerdict: Arrow compute kernels work but are verbose.")
print("PyArrow is the foundation layer — you'd typically use Polars or DuckDB")
print("on top rather than writing raw Arrow compute chains.")

=== pyarrow with compute kernels ===
  client_id     type=string   sample: '349486'
  mes_clean     type=string   sample: '202604'
  collections   type=double  sample: 10400.0

Verdict: Arrow compute kernels work but are verbose.
PyArrow is the foundation layer — you'd typically use Polars or DuckDB
on top rather than writing raw Arrow compute chains.


## 6. What changes in a data lake?

The comparison above tests the **ingestion boundary** — reading raw CSV.
In a cloud data lake, data lives in **columnar formats** (Parquet, Delta,
Iceberg) where the schema is baked into the file metadata.

Let's demonstrate: once the data is in Parquet, every engine reads the
same types — no inference needed.

In [12]:
from fudo.utils.io import INTERIM_DIR

parquet_path = INTERIM_DIR / "fudo_revenue_clean.parquet"
assert parquet_path.exists(), "Run notebook 01 first to generate the clean parquet"

print("=== Reading the SAME parquet file with each engine ===\n")

# pandas (via pyarrow backend)
df_pq_pandas = pd.read_parquet(parquet_path)
print("pandas from parquet:")
for c in df_pq_pandas.columns:
    print(f"  {c:25s}  {str(df_pq_pandas[c].dtype)}")

print()

# polars
df_pq_polars = pl.read_parquet(parquet_path)
print("polars from parquet:")
for c in df_pq_polars.columns:
    print(f"  {c:25s}  {str(df_pq_polars[c].dtype)}")

print()

# duckdb
duck_pq = con.sql(f"SELECT * FROM '{parquet_path}' LIMIT 1")
print("duckdb from parquet:")
for col_name, col_type in zip(duck_pq.columns, duck_pq.types):
    print(f"  {col_name:25s}  {str(col_type)}")

=== Reading the SAME parquet file with each engine ===



pandas from parquet:
  client_id                  str
  transaction_date           str
  currency                   str
  collections                float64
  product                    str
  pais                       str
  fecha_primer_pago          datetime64[us]

polars from parquet:
  client_id                  String
  transaction_date           String
  currency                   String
  collections                Float64
  product                    String
  pais                       String
  fecha_primer_pago          Datetime(time_unit='us', time_zone=None)



duckdb from parquet:
  client_id                  VARCHAR
  transaction_date           VARCHAR
  currency                   VARCHAR
  collections                DOUBLE
  product                    VARCHAR
  pais                       VARCHAR
  fecha_primer_pago          TIMESTAMP


## 7. Lazy evaluation — what a data lake query actually looks like

In a real data lake, you don't `read_csv` — you point the engine at
partitioned Parquet/Delta files and push predicates down. The engine
reads only the columns and row groups it needs.

Below: Polars `scan_parquet` (lazy) and DuckDB SQL both demonstrate
predicate pushdown — they never load the full dataset into memory.

In [13]:
# Polars lazy scan — only materializes when .collect() is called
lazy_result = (
    pl.scan_parquet(parquet_path)
    .filter(pl.col("currency") == "CLP")
    .group_by("transaction_date")
    .agg(pl.col("collections").sum().alias("total_clp"))
    .sort("transaction_date")
)

print("Polars query plan (lazy, predicate pushdown):")
print(lazy_result.explain())
print("\nResult:")
lazy_result.collect()

Polars query plan (lazy, predicate pushdown):


SORT BY [col("transaction_date")]
  AGGREGATE[maintain_order: false]
    [col("collections").sum().alias("total_clp")] BY [col("transaction_date")]
    FROM
    simple π 2/2 ["collections", ... 1 other column]
      Parquet SCAN [/home/alfonso/projects/fudo/data/interim/fudo_revenue_clean.parquet]
      PROJECT 3/7 COLUMNS
      SELECTION: [(col("currency")) == ("CLP")]
      ESTIMATED ROWS: 33959

Result:


transaction_date,total_clp
str,f64
"""2026-01""",1.2976e8
"""2026-02""",1.6663e8
"""2026-03""",1.6807e8
"""2026-04""",1.2997e8


In [14]:
# DuckDB SQL — reads parquet directly, pushes filters into the scan
con.sql(f"""
    SELECT transaction_date, SUM(collections) AS total_clp
    FROM '{parquet_path}'
    WHERE currency = 'CLP'
    GROUP BY transaction_date
    ORDER BY transaction_date
""").show()

┌──────────────────┬────────────────────┐
│ transaction_date │     total_clp      │
│     varchar      │       double       │
├──────────────────┼────────────────────┤
│ 2026-01          │ 129758800.57000001 │
│ 2026-02          │ 166628422.34999993 │
│ 2026-03          │ 168071382.85999998 │
│ 2026-04          │ 129968640.38000004 │
└──────────────────┴────────────────────┘



## 8. Takeaways

### Auto-inference: universal failure

**No engine resolved the Metabase comma problem automatically.** All four
(pandas, PyArrow, Polars, DuckDB) inferred every column as string/VARCHAR.

The root cause is structural: the CSV uses commas as the field delimiter
*and* as thousand separators inside quoted values. This ambiguity is
irresolvable without domain knowledge — no sniffer can distinguish
`"349,486"` (an ID with a comma) from `"10,400"` (a number with a
thousand separator) by inspecting bytes alone.

### Hint-assisted ingestion: where engines differ

| Engine | Ergonomics | Approach |
|--------|-----------|----------|
| **pandas** | Weakest | `thousands=","` is global — no per-column control. Manual `.str.replace().astype()` is the practical path. |
| **PyArrow** | Verbose | Arrow compute kernels (`utf8_replace_substring` → `cast`) work but feel like assembly language. |
| **Polars** | Best for pipelines | Expression API (`.str.replace_all().cast()`) is composable, column-specific, and works in both eager and lazy mode. |
| **DuckDB** | Best for SQL teams | `REPLACE() + CAST()` in SQL is concise and familiar. One-liner ingestion: `COPY (SELECT ...) TO 'lake.parquet'`. |

### After ingestion (data lake reads)

**The dtype inference problem disappears.** Parquet, Delta, and Iceberg
store the schema in file metadata. Every engine reads the same types —
no inference needed. Section 6 above demonstrates this.

### Implications for this pipeline

1. **The real fix is at the source.** Fudo should export CSV without
   thousand separators — or better, export Parquet directly from Metabase.
2. **For the ingestion boundary (CSV → lake), DuckDB or Polars** offer
   the cleanest transform-at-read ergonomics. DuckDB if you want SQL,
   Polars if you want a DataFrame API.
3. **For downstream analytics, Polars or DuckDB replace pandas** with
   lazy evaluation, predicate pushdown, and better memory efficiency.
   Section 7 demonstrates this with the same aggregation query.
4. **PySpark** is relevant at scale (millions+ rows, distributed cluster).
   For 34K rows it adds infrastructure complexity without benefit. However,
   if Fudo's data grows to multi-GB, Spark on a lake (Delta/Iceberg)
   reads pre-typed Parquet — same zero-inference benefit.
5. **Schema-on-write beats schema-on-read.** Once data enters the lake in
   a typed columnar format, the engine choice stops mattering for dtype
   correctness — it only matters for query performance.
